*   https://unstats.un.org/unsd/snaama
*   https://unstats.un.org/unsd/snaama/Basic (Basic Data Selection → GDP, Annual Rate of Growth - Percentage)

In [1]:
import pandas as pd

# Load the dataset
file_path = '/content/Results.csv'
df = pd.read_csv(file_path)

# Display the first few rows in a table format
df.head()

,Country/Area,Year,Unit,"GDP, Annual Rate of Growth - Percentage"
0,Afghanistan,1970,...,...
1,Afghanistan,1971,Percentage,-0.2757095194978465
2,Afghanistan,1972,Percentage,-16.401219343449945
3,Afghanistan,1973,Percentage,2.8958465599867367
4,Afghanistan,1974,Percentage,5.453020134203168


In [2]:
# Assuming columns might be named 'Country', 'Year', and 'GDP Annual Growth Rate (%)' or similar.
# Let's write a flexible parser to find relevant columns.
country_col = [col for col in df.columns if 'country' in col.lower() or 'name' in col.lower()][0]
year_col = [col for col in df.columns if 'year' in col.lower() or 'date' in col.lower()][0]
gdp_col = [col for col in df.columns if 'gdp' in col.lower() and ('growth' in col.lower() or 'rate' in col.lower())][0]

# Drop rows with missing values in essential columns
df_clean = df.dropna(subset=[country_col, year_col, gdp_col]).copy()

# Convert Year and GDP Growth columns to numeric if they aren't already
df_clean[year_col] = pd.to_numeric(df_clean[year_col], errors='coerce')
df_clean[gdp_col] = pd.to_numeric(df_clean[gdp_col], errors='coerce')
df_clean = df_clean.dropna(subset=[year_col, gdp_col])

# Find the most recent year
most_recent_year = int(df_clean[year_col].max())

# Filter for the most recent year
df_recent = df_clean[df_clean[year_col] == most_recent_year]

# Rank countries by GDP growth rate in descending order
ranked_df = df_recent[[country_col, gdp_col]].sort_values(by=gdp_col, ascending=False).reset_index(drop=True)

# Create a structured presentation table with 'Rank'
ranked_table = pd.DataFrame({
    'Rank': range(1, len(ranked_df) + 1),
    'Country/Area': ranked_df[country_col],
    'GDP Annual Growth Rate (%)': ranked_df[gdp_col].round(2)
})

# Display the table cleanly without displaying the default DataFrame index
from IPython.display import HTML
HTML(ranked_table.to_html(index=False))

Rank,Country/Area,GDP Annual Growth Rate (%)
1,Guyana,43.57
2,Palau,12.00
3,Comoros,10.64
4,Georgia,9.68
5,Guinea,9.25
6,Kyrgyzstan,9.04
7,Monaco,8.83
8,Niger,8.48
9,Tajikistan,8.36
10,"China, Macao Special Administrative Region",7.85


In [3]:
import pandas as pd
from IPython.display import HTML, display

# Ensure we are using the cleaned dataframe with valid country names
available_countries = df_clean[country_col].unique()

selected_country = None

while True:
    search_query = input("Enter a country name (or part of it) to retrieve historical GDP growth (or type 'exit' to cancel): ").strip()
    if not search_query:
        print("Please enter a valid search term.")
        continue

    # Allow the user to exit
    if search_query.lower() in ['exit', 'quit']:
        print("Search cancelled by the user.")
        break

    # Find potential matches (case-insensitive)
    matches = [c for c in available_countries if search_query.lower() in c.lower()]

    if not matches:
        print(f"No countries found matching '{search_query}'. Please try again.\n")
    elif len(matches) == 1:
        selected_country = matches[0]
        print(f"\nShowing full historical data for: {selected_country}")
        break
    else:
        # Display the entire list of matches as a vertical list
        print(f"\nMultiple matches found ({len(matches)}):")
        for m in matches:
            print(f"- {m}")
        print("Please be more specific with your search term.\n")

# Filter and display the selected country's history if a selection was made
if selected_country:
    country_data = df_clean[df_clean[country_col] == selected_country].copy()
    country_data_sorted = country_data.sort_values(by=year_col)

    # Format table for output, including a sequential numbered list
    presentation_df = pd.DataFrame({
        'No.': range(1, len(country_data_sorted) + 1),
        'Year': country_data_sorted[year_col].astype(int),
        'GDP Annual Growth Rate (%)': country_data_sorted[gdp_col].round(2)
    })

    display(HTML(presentation_df.to_html(index=False)))

Enter a country name (or part of it) to retrieve historical GDP growth (or type 'exit' to cancel): switzerland

Showing full historical data for: Switzerland


No.,Year,GDP Annual Growth Rate (%)
1,1971,4.08
2,1972,3.20
3,1973,3.05
4,1974,1.45
5,1975,-7.28
6,1976,-1.40
7,1977,2.43
8,1978,0.41
9,1979,2.49
10,1980,4.60


In [4]:
import numpy as np
import pandas as pd
from IPython.display import display

# 1-Year CAGR is simply the annual growth rate itself for the most recent year.
# Let's find the most recent year in the cleaned dataset.
latest_year = int(df_clean[year_col].max())

df_1yr = df_clean[df_clean[year_col] == latest_year].copy()

# To calculate the 5-Year CAGR, we need 5 consecutive years of growth data ending in the most recent year.
start_year = latest_year - 4

# Filter data for this 5-year window
df_5yr_window = df_clean[(df_clean[year_col] >= start_year) & (df_clean[year_col] <= latest_year)].copy()

# We need to make sure countries have data for all 5 years in this window
years_per_country = df_5yr_window.groupby(country_col)[year_col].nunique()
valid_countries = years_per_country[years_per_country == 5].index

df_5yr_filtered = df_5yr_window[df_5yr_window[country_col].isin(valid_countries)]

# Calculate CAGR by compounding the annual growth rates:
# CAGR = [ Product(1 + Growth_Rate_i / 100) ]^(1/5) - 1
def calculate_cagr(group):
    compounded_multiplier = np.prod(1 + group[gdp_col] / 100.0)
    if compounded_multiplier <= 0:
        return np.nan
    cagr = (compounded_multiplier ** (1 / 5.0) - 1) * 100
    return cagr

cagr_5yr = df_5yr_filtered.groupby(country_col).apply(calculate_cagr, include_groups=False).reset_index(name='5-Year CAGR (%)')

# Sort both datasets to find rankings
ranked_1yr = df_1yr[[country_col, gdp_col]].sort_values(by=gdp_col, ascending=False).reset_index(drop=True)
ranked_1yr.columns = ['Country/Area', '1-Year Growth Rate (%)']

ranked_5yr = cagr_5yr.dropna().sort_values(by='5-Year CAGR (%)', ascending=False).reset_index(drop=True)
ranked_5yr.columns = ['Country/Area', '5-Year CAGR (%)']

# Configure pandas to display all rows and columns without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("--- All Countries by 1-Year Growth Rate ---")
display(ranked_1yr)

print("\n--- All Countries by 5-Year CAGR ---")
display(ranked_5yr)

# Reset pandas options back to default afterwards
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

--- All Countries by 1-Year Growth Rate ---


,Country/Area,1-Year Growth Rate (%)
0,Guyana,43.572451
1,Palau,12.004298
2,Comoros,10.637849
3,Georgia,9.679794
4,Guinea,9.249334
5,Kyrgyzstan,9.040136
6,Monaco,8.830038
7,Niger,8.477751
8,Tajikistan,8.357000
9,"China, Macao Special Administrative Region",7.849602



--- All Countries by 5-Year CAGR ---


,Country/Area,5-Year CAGR (%)
0,Guyana,40.137888
1,Guinea,12.402073
2,Tajikistan,7.691345
3,Senegal,6.410527
4,Democratic Republic of the Congo,6.387961
5,Georgia,6.352859
6,Monaco,6.249956
7,Comoros,6.226511
8,Benin,6.187618
9,Ethiopia,6.184351
